# Pretraining nanoGPT on TinyShakespeare

This notebook assembles everything built so far into a complete, instrumented pretraining loop. The three ingredients that separate a loop that merely works from one that is compute-optimal are mixed precision arithmetic, a principled learning rate schedule, and an understanding of scaling laws.

**Mixed precision** — BF16 vs FP16, `torch.autocast`, `GradScaler`, and why you should almost always use BF16 on modern hardware. The argument is numerical, not just a recommendation: BF16 shares FP32's 8-bit exponent and therefore has the same dynamic range; it can never overflow, requires no loss scaling, and frees the optimizer moments to stay in FP32 without any extra machinery.

**Learning rate schedule** — linear warmup plus cosine decay, derived from the structure of the Adam optimizer. Warmup is not a heuristic: at step 1, Adam's bias correction amplifies the first gradient by $1/(1 - \beta_1)$; warmup keeps the early steps small until the moment estimates are reliable. Cosine decay has the right shape — slow early, fast through the middle, long flat tail — and its `min_lr` parameter controls how much useful learning continues until the last step.

**Scaling laws** — the Chinchilla result $D^* \approx 20N$. Given a fixed compute budget in FLOPs, the loss surface $L(N, D) = E + A/N^\alpha + B/D^\beta$ yields an optimal allocation between model size and token count. We use this to plan and evaluate training runs before writing a single line of the loop.

## Mixed Precision Training

### The floating-point formats

A standard FP32 number uses 32 bits: 1 sign, 8 exponent, 23 mantissa bits (~7 decimal digits of precision).

| Format | Bits | Exponent | Mantissa | Max value | Precision |
|---|---|---|---|---|---|
| FP32   | 32  | 8        | 23       | 3.4e38    | ~7 decimal digits |
| FP16   | 16  | 5        | 10       | 65504     | ~3 decimal digits |
| BF16   | 16  | 8        | 7        | 3.4e38    | ~2 decimal digits |

: {tbl-colwidths="[15,10,15,15,20,25]"}

FP16 has a maximum representable value of 65504. Activations and gradients in a deep network can easily exceed this — especially early in training when the loss is large. Overflow produces `inf`, which propagates to `nan` and kills the run. [BF16 has the same 8-bit exponent as FP32, giving it the same dynamic range ($3.4 \times 10^{38}$). It sacrifices mantissa precision but can never overflow.]{.mark} For neural network training, where values span many orders of magnitude but sub-1% precision rarely matters, BF16 is strictly better than FP16 on hardware that supports it (A100, H100, recent consumer cards):

In [ ]:
import torch

# FP16 overflow
x = torch.tensor(70000.0)
print(x.to(torch.float16))   # tensor(inf)    — overflow!
print(x.to(torch.bfloat16))  # tensor(69632.) — representable, slightly rounded

# FP16 gradient underflow
small_grad = torch.tensor(1e-8)
print(small_grad.to(torch.float16))   # tensor(0.)         — underflow to zero!
print(small_grad.to(torch.bfloat16))  # tensor(1.0000e-08) — preserved

### `torch.autocast`

`torch.autocast` is a context manager that casts operations to lower precision automatically. It is *selective* — operations sensitive to precision (softmax, layer norm) stay in FP32; operations that benefit from low-precision throughput (matmul, conv) are cast to BF16/FP16. Model parameters remain FP32 throughout; autocast only affects intermediate computation within the forward pass:

In [ ]:
import contextlib
import torch.nn as nn

# Determine dtype and device at startup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# Build a context that is a no-op on CPU (CPU has no BF16 tensor cores)
autocast_ctx = (
    torch.autocast(device_type='cuda', dtype=dtype)
    if torch.cuda.is_available()
    else contextlib.nullcontext()
)

# BF16 training step — no GradScaler needed
# with autocast_ctx:
#     logits, loss = model(x, y)
# loss.backward()
# torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
# optimizer.step()
print(f"device={device}  dtype={dtype}")

### `GradScaler` for FP16

If you must use FP16 (older hardware with no BF16 support), `GradScaler` prevents gradient underflow. It multiplies the loss by a large scalar $S$ before backward — scaling all gradients up by $S$ to keep them representable in FP16 — then divides them back by $S$ before the optimizer step. The scaler doubles $S$ every 2000 unproblematic steps and halves it whenever a step is skipped due to overflow:

In [ ]:
from torch.cuda.amp import GradScaler

# FP16 training step
scaler = GradScaler()   # starts with scale=65536

# for x, y in dataloader:
#     with torch.autocast(device_type='cuda', dtype=torch.float16):
#         logits, loss = model(x, y)
#
#     scaler.scale(loss).backward()
#
#     # Unscale before clipping — clip_grad_norm_ must see true magnitudes
#     scaler.unscale_(optimizer)
#     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#
#     # Skips if gradients contain inf/nan; updates scale factor
#     scaler.step(optimizer)
#     scaler.update()
print("GradScaler ready — scale:", scaler.get_scale())

:::{.callout-note}
## BF16 vs FP16 decision rule
If your GPU supports BF16 (A100, H100, RTX 3090+, 4090), always use BF16. You get the same speed benefit as FP16 with none of the overflow risk and no `GradScaler` boilerplate. Check with `torch.cuda.is_bf16_supported()`.

:::

## Learning Rate Schedules

### Why warmup?

Adam maintains running estimates of the gradient mean ($m_t$) and variance ($v_t$) for each parameter:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

At step 1, $m_0 = v_0 = 0$. The bias-corrected estimates are:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

[At $t=1$ with $\beta_1=0.9$, $\hat{m}_1 = m_1 / 0.1 = 10 m_1$ — the first gradient is amplified 10× by the bias correction.]{.mark} The effective step size is large and unreliable until $m_t$ and $v_t$ have accumulated enough history. Warmup counteracts this: by starting with LR $\approx 0$ and linearly increasing it over the first $T_{\text{warm}}$ steps, we give the optimizer time to build reliable momentum estimates before taking large steps.

**How many warmup steps?** Rule of thumb: 1–2% of total training steps. For a 5000-step run: 50–100 warmup steps.

### Why cosine decay?

Early in training the model is far from a minimum — large steps navigate the loss landscape quickly. Late in training the model is near a minimum — large steps risk overshooting. [Cosine decay has the right shape: slow decrease early, rapid decrease through the middle, and a long flat tail approaching `min_lr`.]{.underline}

The combined schedule:

$$\text{lr}(t) = \begin{cases}
\text{max\_lr} \cdot \dfrac{t}{T_{\text{warm}}} & t < T_{\text{warm}} \\[1em]
\text{min\_lr} + \dfrac{\text{max\_lr} - \text{min\_lr}}{2}\left(1 + \cos\dfrac{\pi(t - T_{\text{warm}})}{T_{\text{max}} - T_{\text{warm}}}\right) & t \geq T_{\text{warm}}
\end{cases}$$

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR


def make_cosine_schedule(
    optimizer,
    max_lr:       float,
    min_lr:       float,
    warmup_steps: int,
    max_steps:    int,
) -> LambdaLR:
    """
    Linear warmup from 0 to max_lr over warmup_steps,
    then cosine decay from max_lr to min_lr over max_steps.

    Set the optimizer's initial lr = max_lr.
    """
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(warmup_steps, 1)          # linear warmup: 0 → 1.0
        progress = (step - warmup_steps) / max(max_steps - warmup_steps, 1)
        progress = min(progress, 1.0)                   # clamp after max_steps
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        min_ratio = min_lr / max_lr
        return min_ratio + (1.0 - min_ratio) * cosine   # scale to [min_ratio, 1.0]
    return LambdaLR(optimizer, lr_lambda)

We visualize the schedule for the nano model configuration — `max_lr=3e-4`, `min_lr=3e-5` (10% of max, the Chinchilla convention), 100 warmup steps, 5000 total steps:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt

MAX_LR    = 3e-4
MIN_LR    = 3e-5   # 10% of max — Chinchilla convention
WARMUP    = 100
MAX_STEPS = 5000

dummy_param = torch.nn.Parameter(torch.zeros(1))
opt   = torch.optim.AdamW([dummy_param], lr=MAX_LR)
sched = make_cosine_schedule(opt, MAX_LR, MIN_LR, WARMUP, MAX_STEPS)

lrs = []
for _ in range(MAX_STEPS):
    lrs.append(opt.param_groups[0]['lr'])
    sched.step()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs, color='#9C27B0', lw=2)
ax.axvline(WARMUP, color='gray', linestyle='--', lw=0.8,
           label=f'warmup ends (step {WARMUP})')
ax.axhline(MIN_LR, color='gray', linestyle=':', lw=0.8,
           label=f'min_lr={MIN_LR:.1e}')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine Schedule with Linear Warmup')
ax.legend()
plt.tight_layout()
plt.show()

The `min_lr = 0.1 × max_lr` choice keeps updates meaningful to the last step. A smaller `min_lr` (e.g., `1e-8`) wastes the last 20% of training — weights barely move. A larger `min_lr` (e.g., `0.5 × max_lr`) means the schedule decays insufficiently and the model does not converge as tightly.

## Scaling Laws

### The Chinchilla result

Hoffmann et al. (2022) trained hundreds of language models of different sizes on different amounts of data and fit a parametric model to the loss surface:

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

where $N$ is the number of model parameters, $D$ is the number of training tokens, $E \approx 1.69$ is the irreducible entropy of natural language, and $A, B, \alpha, \beta$ are fitted constants.

Given a compute budget $C \approx 6ND$ FLOPs (6 FLOPs per parameter per token for a forward and backward pass[^flops]), the compute-optimal allocation is:

$$N^* \propto C^{0.5}, \qquad D^* \propto C^{0.5}$$

which yields the famous rule of thumb:

$$\boxed{D^* \approx 20 \cdot N}$$

[^flops]: The $6N$ FLOPs per token approximation counts 2 FLOPs per multiply-add in the forward pass (giving $2N$) and doubles this for the backward pass ($4N$), totaling $6N$ FLOPs per training token. This is a rough count that ignores attention FLOPs (which scale with sequence length) and is accurate to within a factor of 2 for most transformer configurations.

In [ ]:
def chinchilla_optimal(compute_flops: float) -> dict:
    """
    Given a compute budget in FLOPs, return the compute-optimal
    model size (N*) and token count (D*).
    Uses D* ≈ 20N* from the Chinchilla scaling law.
    """
    # C ≈ 6 * N * D → at D* = 20*N*: C = 6 * N* * 20 * N* → N* = sqrt(C / 120)
    optimal_n = (compute_flops / 120) ** 0.5
    optimal_d = 20 * optimal_n
    return {
        'compute_flops':    compute_flops,
        'compute_pflops':   compute_flops / 1e15,
        'optimal_params':   optimal_n,
        'optimal_params_M': optimal_n / 1e6,
        'optimal_tokens':   optimal_d,
        'optimal_tokens_B': optimal_d / 1e9,
    }


def flops_per_step(n_params: int, batch_tokens: int) -> float:
    """Approximate FLOPs for one training step: 6 * N * batch_tokens."""
    return 6 * n_params * batch_tokens


def plan_training_run(
    n_params:        int,
    gpu_flops_per_s: float,
    gpu_count:       int,
    hours:           float,
    batch_tokens:    int,
) -> dict:
    """
    Given hardware and time budget, print the training plan and
    compare against Chinchilla-optimal allocation.
    """
    total_flops   = gpu_flops_per_s * gpu_count * hours * 3600
    flops_per_stp = flops_per_step(n_params, batch_tokens)
    total_steps   = int(total_flops / flops_per_stp)
    total_tokens  = total_steps * batch_tokens
    chinchilla    = chinchilla_optimal(total_flops)
    token_ratio   = total_tokens / chinchilla['optimal_tokens']

    print(f"\nTraining Run Plan")
    print(f"{'─'*52}")
    print(f"  Model parameters:     {n_params/1e6:.1f}M")
    print(f"  Hardware:             {gpu_count}× GPU @ {gpu_flops_per_s/1e12:.0f} TFLOP/s")
    print(f"  Time budget:          {hours:.1f} hours")
    print(f"  Total FLOPs:          {total_flops/1e18:.2f} EFLOPs")
    print(f"  Total steps:          {total_steps:,}")
    print(f"  Total tokens:         {total_tokens/1e6:.0f}M")
    print(f"{'─'*52}")
    print(f"  Chinchilla-optimal N: {chinchilla['optimal_params_M']:.1f}M params")
    print(f"  Chinchilla-optimal D: {chinchilla['optimal_tokens_B']:.2f}B tokens")
    print(f"  Our token ratio:      {token_ratio:.2f}× optimal")
    if token_ratio < 0.1:
        print(f"  ⚠ Severely undertrained — consider a smaller model")
    elif token_ratio < 0.5:
        print(f"  ⚠ Undertrained — model could learn more with more data")
    elif token_ratio > 2.0:
        print(f"  ⚠ Overtrained on this dataset — model may be memorizing")
    else:
        print(f"  ✓ Near compute-optimal")
    return {'total_steps': total_steps, 'total_tokens': total_tokens, 'token_ratio': token_ratio}


# Our 29.9M nano model on a consumer GPU
plan = plan_training_run(
    n_params=29_900_000,    # 29.9M — actual count with SwiGLU FFN (see NB01)
    gpu_flops_per_s=20e12,  # RTX 3090: ~20 TFLOP/s BF16
    gpu_count=1,
    hours=1.0,
    batch_tokens=8 * 256,   # batch_size=8, seq_len=256 → 2048 tokens/step
)

Our 29.9M nano model is Chinchilla-optimal at $D^* = 20 \times 29.9\text{M} = 598\text{M}$ tokens. TinyShakespeare has ~1M tokens — about 0.2% of what we need for a compute-optimal run. This means: (1) the model will overfit TinyShakespeare after a few epochs; (2) for a real compute-optimal run we need a much larger corpus; (3) the training runs in this series demonstrate the mechanics and tooling, not production-quality generalization.

:::{.callout-caution}
## The param count matters
The source tutorials cite `10.7M parameters` for the nano GPT. That count used a plain GELU FFN with `d_ff = 4d`. We use SwiGLU with `d_ff = int(2 * 4 * d / 3)` rounded to the nearest 64, which gives a different FFN size and a total of **29.9M** parameters. All Chinchilla planning must use the correct count.

:::

## The Complete Pretraining Loop

Everything assembled: BF16 autocast, gradient accumulation (from [NB04](/courses/llm/04-training-loop.html)), cosine schedule, gradient monitoring (from [NB05](/courses/llm/05-scaling-diagnostics.html)), data pipeline (from [NB03](/courses/llm/03-data-pipelines.html)), and structured logging.

The `TrainingConfig` dataclass centralizes all hyperparameters:

In [ ]:
import numpy as np
import time
from pathlib import Path
from dataclasses import dataclass, field

from notebook_01 import GPT, NanoGPTConfig   # architecture from NB01
from notebook_02 import Tokenizer            # BPE tokenizer from NB02
from notebook_03 import PretrainingDataset, make_dataloader  # pipeline from NB03
from notebook_05 import TrainingLogger       # logger from NB05


@dataclass
class TrainingConfig:
    # Model
    model_config:     NanoGPTConfig = None

    # Optimization
    max_lr:           float = 3e-4
    min_lr:           float = 3e-5   # 10% of max_lr — Chinchilla convention
    weight_decay:     float = 0.1
    beta1:            float = 0.9
    beta2:            float = 0.95
    grad_clip:        float = 1.0

    # Schedule
    warmup_steps:     int   = 100
    max_steps:        int   = 5000

    # Batch
    batch_size:       int   = 8
    accumulation:     int   = 1    # gradient accumulation micro-steps

    # Data
    block_size:       int   = 256
    num_workers:      int   = 2

    # Logging & checkpointing
    eval_every:       int   = 500
    eval_batches:     int   = 20
    log_every:        int   = 10
    checkpoint_every: int   = 1000

    # Paths
    data_dir:   str = 'data/shakespeare'
    run_dir:    str = 'runs/nano_gpt'

    def __post_init__(self):
        if self.model_config is None:
            self.model_config = NanoGPTConfig()

The `compute_grad_stats` helper reads `.grad` directly after `.backward()` — no hooks needed — and returns the global gradient norm plus the per-layer ratio statistics used by the logger and `HealthReport`:

In [ ]:
def compute_grad_stats(model) -> tuple[float, float, float, float]:
    """
    Compute global gradient norm and per-layer ratio statistics in one pass.
    Returns (gnorm, mean_ratio, min_ratio, max_ratio).
    Call after loss.backward() and before optimizer.step().
    """
    total_sq, ratios = 0.0, []
    for mod in model.modules():
        if isinstance(mod, nn.Linear) and mod.weight.grad is not None:
            g = mod.weight.grad.norm().item()
            w = mod.weight.norm().item()
            total_sq += g ** 2
            ratios.append(g / (w + 1e-8))
    gnorm = total_sq ** 0.5
    if not ratios:
        return gnorm, 0.0, 0.0, 0.0
    return gnorm, float(np.mean(ratios)), float(np.min(ratios)), float(np.max(ratios))

The main `pretrain` function:

1. Separates weight-decay parameters (2-D weight matrices) from no-decay parameters (biases and RMSNorm scales) — biases and norm parameters should never be regularized.
2. Prints the Chinchilla plan before training so you know whether the run is compute-optimal.
3. Uses `torch.autocast` for BF16 on CUDA.
4. Accumulates gradients across `cfg.accumulation` micro-steps before each optimizer step.
5. Evaluates on the validation split every `eval_every` steps and saves the best checkpoint.
6. Saves periodic checkpoints that include model, optimizer, and scheduler state.

In [ ]:
def pretrain(cfg: TrainingConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    print(f"Device: {device}  |  dtype: {dtype}")

    # ---- Model ----
    model    = GPT(cfg.model_config).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params/1e6:.1f}M parameters")

    # ---- Optimizer ----
    # Weight matrices: apply weight decay.
    # Biases + RMSNorm scales (1-D tensors): no weight decay.
    decay_params   = [p for n, p in model.named_parameters() if p.dim() >= 2]  # <1>
    nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([
        {'params': decay_params,   'weight_decay': cfg.weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0},
    ], lr=cfg.max_lr, betas=(cfg.beta1, cfg.beta2))
    print(f"Optimizer: {len(decay_params)} decay params, {len(nodecay_params)} no-decay params")

    # ---- Schedule ----
    scheduler = make_cosine_schedule(
        optimizer, cfg.max_lr, cfg.min_lr, cfg.warmup_steps, cfg.max_steps
    )

    # ---- Data ----
    tok = Tokenizer.load('nano_tokenizer.json')
    train_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size, split='train', buffer_size=500
    )
    val_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size, split='val', buffer_size=100
    )
    train_loader = make_dataloader(train_ds, cfg.batch_size, cfg.num_workers)
    val_loader   = make_dataloader(val_ds, cfg.batch_size, 1)

    # ---- Logging ----
    logger = TrainingLogger(run_dir=cfg.run_dir, run_name='nano_gpt_pretrain')

    # ---- Chinchilla plan ----
    batch_tokens = cfg.batch_size * cfg.block_size * cfg.accumulation
    plan_training_run(
        n_params=n_params,
        gpu_flops_per_s=20e12,
        gpu_count=1,
        hours=cfg.max_steps * batch_tokens / (20e12 * 3600),
        batch_tokens=batch_tokens,
    )

    # ---- Training loop ----
    Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
    autocast_ctx = (
        torch.autocast(device_type='cuda', dtype=dtype)
        if torch.cuda.is_available()
        else contextlib.nullcontext()
    )

    model.train()
    total_tokens = 0
    step         = 0
    best_eval    = float('inf')
    train_iter   = iter(train_loader)

    while step < cfg.max_steps:
        t0 = time.time()

        # ---- Gradient accumulation ----
        optimizer.zero_grad()
        step_loss = 0.0

        for micro_step in range(cfg.accumulation):  # <2>
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)

            x, y = x.to(device), y.to(device)

            with autocast_ctx:
                _, loss = model(x, y)

            loss = loss / cfg.accumulation  # <3>
            loss.backward()
            step_loss    += loss.item()
            total_tokens += x.numel()

        # ---- Gradient stats (before clip) ----
        gnorm, mean_r, min_r, max_r = compute_grad_stats(model)

        # ---- Clip and step ----
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        # ---- Throughput ----
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - t0
        tps     = x.numel() / elapsed
        lr      = optimizer.param_groups[0]['lr']

        # ---- Log ----
        if step % cfg.log_every == 0:
            logger.log_step(
                step=step,
                train_loss=step_loss,
                learning_rate=lr,
                global_grad_norm=gnorm,
                total_tokens=total_tokens,
                mean_grad_ratio=mean_r,
                min_grad_ratio=min_r,
                max_grad_ratio=max_r,
            )

        # ---- Eval ----
        if step % cfg.eval_every == 0:
            model.eval()
            eval_losses = []
            with torch.no_grad():
                for i, (xv, yv) in enumerate(val_loader):
                    if i >= cfg.eval_batches:
                        break
                    with autocast_ctx:
                        _, vl = model(xv.to(device), yv.to(device))
                    eval_losses.append(vl.item())
            eval_loss = float(np.mean(eval_losses))
            model.train()

            logger.log_step(
                step=step,
                train_loss=step_loss,
                learning_rate=lr,
                global_grad_norm=gnorm,
                total_tokens=total_tokens,
                eval_loss=eval_loss,
            )
            print(
                f"step {step:5d}  "
                f"train={step_loss:.4f}  eval={eval_loss:.4f}  "
                f"lr={lr:.2e}  gnorm={gnorm:.3f}  "
                f"ρ=[{min_r:.1e},{max_r:.1e}]"
            )

            if eval_loss < best_eval:
                best_eval = eval_loss
                torch.save({  # <4>
                    'step':      step,
                    'model':     model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'config':    cfg,
                    'eval_loss': eval_loss,
                }, f'{cfg.run_dir}/best_checkpoint.pt')

        # ---- Periodic checkpoint ----
        if step % cfg.checkpoint_every == 0 and step > 0:
            torch.save({
                'step':      step,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
            }, f'{cfg.run_dir}/checkpoint_step{step:05d}.pt')

        step += 1

    logger.close()
    print(f"\nPretraining complete.")
    print(f"  Total tokens: {total_tokens:,}")
    print(f"  Best eval loss: {best_eval:.4f}")
    print(f"  Checkpoint: {cfg.run_dir}/best_checkpoint.pt")
    return model

1. `p.dim() >= 2` selects all weight matrices. Biases and RMSNorm scale parameters are 1-D and should not be regularized — L2 regularization on biases discourages the network from using constant offsets, which is almost never what you want.
2. Each micro-step accumulates gradients without calling `optimizer.step()`. This is equivalent to a single step with a larger batch — the gradients sum before the optimizer sees them.
3. Dividing by `accumulation` before backward normalizes the gradient scale to be independent of the number of micro-steps. Without this, the effective gradient magnitude scales with `accumulation`, requiring you to tune LR every time you change batch size.
4. The checkpoint saves model, optimizer, and scheduler together. [Always save the optimizer state.]{.underline} The Adam moments encode recent gradient history for every parameter. Restarting without them means the first 100–200 steps after a reload behave like fresh training — the optimizer takes large, unstable steps until the moments rebuild.

### Checkpoint format and resume

The checkpoint dictionary contains everything needed to resume training from exactly the state where it was interrupted:

In [ ]:
def load_checkpoint(path: str, model, optimizer, scheduler) -> int:
    """
    Resume training from a checkpoint.
    Returns the step number to continue from.
    """
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])  # <1>
    scheduler.load_state_dict(ckpt['scheduler'])  # <2>
    step = ckpt['step']
    print(f"Resumed from step {step}  eval_loss={ckpt.get('eval_loss', '?')}")
    return step

1. Loading optimizer state restores the Adam $m_t$ and $v_t$ moments for every parameter — the memory of the last several hundred gradient directions.
2. Loading scheduler state restores the current step count inside the LR schedule. Without this, the schedule resets to step 0 and the LR jumps back to the warmup phase.

:::{.callout-caution}
## Stale checkpoints after architecture changes
If you change the model architecture between runs (add a layer, change `d_model`), the checkpoint's optimizer state will not match the new model's parameters. The parameter group sizes will differ and `optimizer.load_state_dict()` will either fail or silently load state for the wrong parameters. Always verify that `len(model.parameters())` matches the number of state tensors in the checkpoint before loading.

:::

## Running a Training Session

With all pieces in place, launching a pretraining session is a single call:

In [ ]:
# cfg = TrainingConfig(
#     max_steps=5000,
#     batch_size=8,
#     accumulation=4,        # effective batch = 32 sequences
#     block_size=256,
#     max_lr=3e-4,
#     min_lr=3e-5,
#     warmup_steps=100,
#     eval_every=500,
#     checkpoint_every=1000,
#     data_dir='data/shakespeare',
#     run_dir='runs/nano_gpt_v1',
# )
# model = pretrain(cfg)

# Post-run analysis from the JSONL log:
# import pandas as pd
# df = pd.DataFrame(TrainingLogger.load_records('runs/nano_gpt_v1/metrics.jsonl'))
# best_step = df.dropna(subset=['eval_loss'])['eval_loss'].idxmin()
# print(f"Best eval loss at step {df.loc[best_step, 'step']}: "
#       f"{df.loc[best_step, 'eval_loss']:.4f}")
print("Training config ready. Call pretrain(cfg) to launch.")

## Summary

| Concept | Key detail |
|---|---|
| FP16 overflow | Max value 65504 — activations can exceed this. Requires `GradScaler`. |
| BF16 dynamic range | Same 8-bit exponent as FP32 — never overflows. No `GradScaler` needed. |
| `torch.autocast` | Casts matmul/conv to BF16; sensitive ops (softmax, RMSNorm) stay FP32. |
| `GradScaler` | For FP16 only: scales loss up before backward, down before optimizer step. |
| Adam warmup reason | Bias correction at $t=1$ amplifies first gradient $1/(1-\beta_1) = 10\times$. |
| Warmup duration | 1–2% of total steps. Too short → early instability; too long → delayed learning. |
| Cosine decay | Slow early, fast through middle, long flat tail. Matches loss landscape curvature. |
| `min_lr` choice | 10% of `max_lr` — keeps updates meaningful to the last step. |
| Chinchilla $D^* = 20N$ | Compute-optimal: 20 tokens per parameter. |
| FLOPs per token | $\approx 6N$ (2× forward + 4× backward). |
| Nano model param count | **29.9M** with SwiGLU FFN — not 10.7M (old GELU count). |
| Decay vs no-decay | Weight matrices: weight decay. Biases + RMSNorm scales: no weight decay. |
| Gradient accumulation | Divide loss by `accumulation` before backward — normalizes gradient scale. |
| Checkpoint contents | Model + optimizer + scheduler state. Load all three to resume correctly. |
| Stale checkpoint | Verify param count matches before loading optimizer state. |

: {tbl-colwidths="[35,65]"}

## Exercises

**1.** Run `pretrain` with `dtype=torch.float32` and `dtype=torch.bfloat16` and compare throughput (tokens/sec). On a CUDA GPU, BF16 should be significantly faster due to tensor core utilization. Report the speedup ratio.

**2.** Plot the LR schedule for three `warmup_steps` values (10, 100, 500) and two `min_lr` values (0, 0.1×max). For each, run 500 training steps and compare loss curves. Confirm that too little warmup causes instability in early steps and `min_lr=0` causes the model to plateau earlier.

**3.** Verify the gradient accumulation equivalence: train for 100 steps with `batch_size=32, accumulation=1` and separately with `batch_size=8, accumulation=4`. Both have effective batch size 32. Compare the final loss and gradient norm trace. They should be nearly identical.

**4.** Use `plan_training_run` to compute the Chinchilla-optimal training duration for the nano model on your machine. Run for exactly that many steps and compare the final eval loss to a run that uses 5× more steps. Confirm that overtraining on TinyShakespeare widens the eval/train gap.

**5.** Implement learning rate **restart** (SGDR): after each full cosine cycle, reset LR to `max_lr` and start a new cosine decay with period 2× the previous period. Compare loss curves between standard cosine and SGDR on 5000 training steps.

**6.** Add a `detect_stale_checkpoint` function that loads a checkpoint and checks whether the optimizer state is consistent with the model: compare the number of parameters in `model.state_dict()` against the number of parameter tensors in `optimizer.state_dict()['state']`. If they differ, print a warning and re-initialize the optimizer instead of loading the stale state.

■